# Python Memoization

## 1. Introduction to Memoization

**What is Memoization?**
- Memoization is an optimization technique that stores the results of expensive function calls
- When the same inputs occur again, the cached result is returned instead of recalculating
- The term comes from "memorandum" (to remember) + "ization"
- It's a form of caching specifically for function results

**Key Benefits:**
- Eliminates redundant computations
- Significantly improves performance for recursive functions
- Trades memory space for faster execution time
- Particularly useful for functions with repeated calculations

**When to Use Memoization:**
✅ Pure functions (same input always produces same output)
✅ Expensive computations
✅ Recursive algorithms with overlapping subproblems
✅ Functions called repeatedly with same arguments

**When NOT to Use Memoization:**
❌ Functions with side effects
❌ Functions with mutable arguments
❌ When memory is constrained
❌ When function is rarely called with same inputs

## 2. The Problem: Redundant Calculations

Let's look at a classic example where memoization makes a huge difference - the Fibonacci sequence.

In [2]:
# Fibonacci WITHOUT memoization - very slow!
import time

def fibonacci_slow(n):
    if n <= 1:
        return n
    return fibonacci_slow(n - 1) + fibonacci_slow(n - 2)

# Test performance
print("Testing fibonacci_slow(35):")
start = time.time()
result = fibonacci_slow(35)
end = time.time()
print(f"Result: {result}")
print(f"Time taken: {end - start:.4f} seconds")

Testing fibonacci_slow(35):
Result: 9227465
Time taken: 6.1928 seconds


## 3. Manual Memoization with Dictionary

The simplest way to implement memoization is using a dictionary to store computed results.

In [ ]:
# Fibonacci WITH manual memoization - much faster!
import time

def fibonacci_manual(n, memo={}):
    if n in memo:
        return memo[n]
    if n <= 1:
        return n
    memo[n] = fibonacci_manual(n - 1, memo) + fibonacci_manual(n - 2, memo)
    return memo[n]

# Test performance
print("Testing fibonacci_manual(35):")
start = time.time()
result = fibonacci_manual(35)
end = time.time()
print(f"Result: {result}")
print(f"Time taken: {end - start:.4f} seconds")

Testing fibonacci_manual(35):
Result: 9227465
Time taken: 0.0001 seconds


In [ ]:
# Let's test with an even larger number
import time

print("Testing fibonacci_manual(100):")
start = time.time()
result = fibonacci_manual(100)
end = time.time()
print(f"Result: {result}")
print(f"Time taken: {end - start:.4f} seconds")

## 4. Using functools.lru_cache (Recommended)

Python's `functools` module provides `lru_cache` - a built-in decorator for memoization.

**LRU = Least Recently Used**
- When the cache is full, the least recently used items are discarded
- `maxsize` parameter controls cache size
- `maxsize=None` means unlimited cache

In [4]:
from functools import lru_cache
import time

# Using lru_cache decorator
@lru_cache(maxsize=None)  # Unlimited cache
def fibonacci_cached(n):
    if n <= 1:
        return n
    return fibonacci_cached(n - 1) + fibonacci_cached(n - 2)

# Test performance
print("Testing fibonacci_cached(100):")
start = time.time()
result = fibonacci_cached(100)
end = time.time()
print(f"Result: {result}")
print(f"Time taken: {end - start:.4f} seconds")

Testing fibonacci_cached(100):
Result: 354224848179261915075
Time taken: 0.0004 seconds


In [ ]:
# Check cache statistics
print("Cache statistics:")
print(f"Cache info: {fibonacci_cached.cache_info()}")
print(f"Cache hits: {fibonacci_cached.cache_info().hits}")
print(f"Cache misses: {fibonacci_cached.cache_info().misses}")

## 5. Custom Memoization Decorator

You can create your own memoization decorator for more control over the caching behavior.

In [ ]:
import functools

def memoize(func):
    """Custom memoization decorator"""
    cache = {}
    
    @functools.wraps(func)
    def wrapper(*args):
        if args not in cache:
            cache[args] = func(*args)
            print(f"Calculating {func.__name__}{args}")
        else:
            print(f"Using cached result for {args}")
        return cache[args]
    
    wrapper.cache = cache  # Expose cache for inspection
    return wrapper

@memoize
def expensive_function(x):
    """Simulate an expensive computation"""
    import time
    time.sleep(0.1)  # Simulate work
    return x ** 2

print("First call (calculates):")
print(expensive_function(5))

print("\nSecond call (uses cache):")
print(expensive_function(5))

print("\nThird call with different argument (calculates):")
print(expensive_function(10))

print("\nFourth call with same argument (uses cache):")
print(expensive_function(10))

## 6. Memoization with Multiple Arguments

Memoization works with functions that have multiple arguments. The cache key is based on all arguments.

In [ ]:
@lru_cache(maxsize=None)
def multiply(a, b):
    print(f"Calculating {a} * {b}")
    return a * b

print("Example with multiple arguments:")
print(f"multiply(3, 4) = {multiply(3, 4)}")
print(f"multiply(3, 4) = {multiply(3, 4)}")  # Uses cache
print(f"multiply(4, 3) = {multiply(4, 3)}")  # Different order, calculates again
print(f"multiply(4, 3) = {multiply(4, 3)}")  # Uses cache

## 7. Memoization with Keyword Arguments

When using keyword arguments, `lru_cache` handles them correctly by creating a consistent cache key.

In [ ]:
@lru_cache(maxsize=None)
def greet(name, greeting="Hello"):
    print(f"Generating greeting for {name}")
    return f"{greeting}, {name}!"

print("Example with keyword arguments:")
print(greet("Alice"))
print(greet("Alice"))  # Uses cache
print(greet("Alice", greeting="Hi"))  # Different argument, calculates
print(greet(name="Alice", greeting="Hi"))  # Uses cache

## 8. Controlling Cache Size

You can limit the cache size to prevent excessive memory usage.

In [5]:
# Limited cache size
@lru_cache(maxsize=3)  # Only store last 3 results
def limited_cache_func(x):
    print(f"Calculating for {x}")
    return x ** 2

print("Testing limited cache (maxsize=3):")
print(limited_cache_func(1))
print(limited_cache_func(2))
print(limited_cache_func(3))
print(limited_cache_func(4))  # Evicts oldest (1)
print(limited_cache_func(1))  # Needs to recalculate

print(f"\nCache info: {limited_cache_func.cache_info()}")

Testing limited cache (maxsize=3):
Calculating for 1
1
Calculating for 2
4
Calculating for 3
9
Calculating for 4
16
Calculating for 1
1

Cache info: CacheInfo(hits=0, misses=5, maxsize=3, currsize=3)


## 9. Clearing the Cache

You can manually clear the cache when needed.

In [ ]:
@lru_cache(maxsize=None)
def cached_function(x):
    print(f"Calculating for {x}")
    return x ** 2

print("Before clearing cache:")
print(cached_function(5))
print(cached_function(5))  # Uses cache
print(f"Cache info: {cached_function.cache_info()}")

# Clear the cache
cached_function.cache_clear()
print("\nAfter clearing cache:")
print(cached_function(5))  # Recalculates
print(f"Cache info: {cached_function.cache_info()}")

## 10. Practical Example: Factorial Calculation

Factorial is another classic example where memoization improves performance.

In [ ]:
# Factorial without memoization
import time

def factorial_slow(n):
    if n <= 1:
        return 1
    return n * factorial_slow(n - 1)

# Factorial with memoization
@lru_cache(maxsize=None)
def factorial_cached(n):
    if n <= 1:
        return 1
    return n * factorial_cached(n - 1)

print("Comparing factorial implementations:")

start = time.time()
result1 = factorial_slow(20)
end = time.time()
print(f"factorial_slow(20): {result1}, Time: {end - start:.6f}s")

start = time.time()
result2 = factorial_cached(20)
end = time.time()
print(f"factorial_cached(20): {result2}, Time: {end - start:.6f}s")

# Second call with cache
start = time.time()
result3 = factorial_cached(20)
end = time.time()
print(f"factorial_cached(20) cached: {result3}, Time: {end - start:.6f}s")

## 11. Practical Example: Expensive Data Processing

Memoization is useful for caching results of expensive data processing operations.

In [ ]:
import hashlib

@lru_cache(maxsize=None)
def expensive_hash(data):
    """Simulate expensive hash computation"""
    print(f"Computing hash for '{data}'")
    # Simulate expensive operation
    for _ in range(1000):
        result = hashlib.sha256(data.encode()).hexdigest()
    return result

print("Example: Caching expensive computations")
print(f"Hash 1: {expensive_hash('hello')[:16]}...")
print(f"Hash 2: {expensive_hash('world')[:16]}...")
print(f"Hash 1 (cached): {expensive_hash('hello')[:16]}...")
print(f"Hash 2 (cached): {expensive_hash('world')[:16]}...")

## 12. Limitations and Caveats

**Important Considerations:**

1. **Memory Usage**: Caching consumes memory. For unlimited caches with many unique inputs, this can be problematic.

2. **Mutable Arguments**: Functions with mutable arguments (lists, dicts) are not safe for memoization because they're not hashable.

3. **Side Effects**: Functions with side effects (printing, writing to files, modifying global state) should not be memoized.

4. **Stale Data**: If the function depends on external state that changes, cached results may become outdated.

In [ ]:
# PROBLEM: Mutable arguments are not hashable
@lru_cache(maxsize=None)
def process_list(lst):
    return sum(lst)

try:
    result = process_list([1, 2, 3])
    print(result)
except TypeError as e:
    print(f"Error: {e}")

# SOLUTION: Convert to tuple first
@lru_cache(maxsize=None)
def process_tuple(tup):
    return sum(tup)

print(f"\nWith tuple: {process_tuple((1, 2, 3))}")

In [ ]:
# PROBLEM: Functions with side effects
counter = 0

@lru_cache(maxsize=None)
def count_calls(x):
    global counter
    counter += 1
    return x * 2

print("Functions with side effects should NOT be memoized:")
print(f"Call 1: {count_calls(5)}, Counter: {counter}")
print(f"Call 2: {count_calls(5)}, Counter: {counter}")  # Counter doesn't increment!
print(f"Call 3: {count_calls(5)}, Counter: {counter}")  # Counter doesn't increment!

## 13. Comparison: Memoization Approaches

| Approach | Pros | Cons | Use Case |
|----------|------|------|----------|
| Manual dict | Simple, full control | Verbose, must pass dict | Learning, simple cases |
| lru_cache | Clean syntax, built-in, cache stats | Limited control | Most common cases |
| Custom decorator | Maximum flexibility | More code to maintain | Special requirements |

## 14. Best Practices

**DO:**
- Use `@lru_cache` for most memoization needs
- Set appropriate `maxsize` to limit memory usage
- Use memoization for pure functions only
- Monitor cache statistics with `cache_info()`
- Clear cache when needed with `cache_clear()`
- Convert mutable arguments to immutable types (tuple, frozenset)

**DON'T:**
- Don't memoize functions with side effects
- Don't use unlimited cache for functions with many unique inputs
- Don't memoize functions that depend on changing external state
- Don't pass mutable arguments directly (lists, dicts)
- Don't forget that memoization uses memory

## 15. Summary

### Key Concepts:

| Concept | Description |
|---------|-------------|
| Memoization | Caching function results to avoid redundant calculations |
| lru_cache | Python's built-in memoization decorator |
| Cache key | Based on function arguments (must be hashable) |
| maxsize | Controls cache size (None = unlimited) |
| cache_info() | Returns cache statistics (hits, misses) |
| cache_clear() | Manually clears the cache |

### When to Use:
- Recursive algorithms (Fibonacci, factorial)
- Expensive computations
- Functions called repeatedly with same inputs
- Pure functions (no side effects)

### When to Avoid:
- Functions with side effects
- Functions with mutable arguments
- When memory is constrained
- When external state changes frequently

### Performance Impact:
- **Without memoization**: O(2^n) for naive Fibonacci
- **With memoization**: O(n) for Fibonacci
- Trade-off: Memory usage vs. computation time

# Chapter 1 Q&A Practice

These questions are imported from `1250 Python Q&A - Copy.xlsx` for Chapter 1. Use them after studying the concept sections above.


## Memoization

### Q37. What is memoization, and how is it implemented in Python?

- **Difficulty:** Medium
- **Estimated effort:** 10–15 minutes
- **Why it matters:** Performance optimization
- **Related / duplicate reference:** Q 67

**Answer outline:** Store previous results to avoid repeated work. Use dictionaries manually or functools.lru_cache for pure/repeated function calls.



## Caching

### Q67. How to use functools.lru_cache for caching?

- **Difficulty:** Medium–Hard
- **Estimated effort:** 20–25 minutes
- **Why it matters:** Performance optimization
- **Related / duplicate reference:** Q 37

**Answer outline:** Store previous results to avoid repeated work. Use dictionaries manually or functools.lru_cache for pure/repeated function calls.

